# 01 · Cleaning and group-aware split
This notebook consumes the audited inventory, applies the approved cleaning rules once, and freezes group-aware train, validation, and test manifests.

In [1]:
from pathlib import Path
import sys, json, hashlib, pandas as pd, numpy as np
sys.path.insert(0,str(Path('../src').resolve()))
from dronedet import splits
import yaml
PROJECT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
CFG = yaml.safe_load((PROJECT/'config.yaml').read_text(encoding='utf-8'))
profile='colab' if 'google.colab' in sys.modules else 'local'
P={k:Path(v) for k,v in CFG['paths'][profile].items()}
def safe(path):
 path=Path(path).resolve(); raw=P['raw_root'].resolve()
 if path==raw or raw in path.parents: raise PermissionError(path)
 path.parent.mkdir(parents=True,exist_ok=True); return path
inv=pd.read_csv(PROJECT/'data/interim/inventory.csv',keep_default_na=False)
m=splits.flag_columns(inv,CFG['cleaning'])
m,reps=splits.apply_cleaning(m,7936)
assert len(reps)==9
assert int((m.exclusion_reason=='too_short').sum())==2
assert int((m.exclusion_reason=='exact_zero_duplicate').sum())==1079
assert int(m.kept.sum())==10623
pool=m[m.kept & ~m.pinned].sort_values('clip_id')
m.loc[m.kept & m.pinned,'split']='train'
m.loc[pool.index,'split']=splits.two_stage_split(pool,int(CFG['seed']))
gates=splits.split_gates(m); display(gates)
assert gates.ok.all(), 'Structural split gate failed; do not freeze or continue.'

,gate,ok,detail
0,S1 group in one split,True,
1,S2 PCM identity in one split,True,
2,S3 both classes in every split,True,
3,S4 kept pool size,True,10623
4,S5 no exact-zero outside train,True,
5,S6 extra_membo_D2 train only,True,56
6,S7 B_S2_D1 blocks,True,
7,S8 pinned groups train only,True,


In [2]:
P_SPLITS=PROJECT/'data/processed/splits'
FROZEN=P_SPLITS/'FROZEN.txt'
HASH_PATH=PROJECT/'results/tables/manifest_hashes.json'
if FROZEN.exists():
    hashes=json.loads(HASH_PATH.read_text(encoding='utf-8'))
    frozen_files={'manifest_full':PROJECT/'data/interim/manifest_full.csv',**{s:P_SPLITS/f'{s}.csv' for s in ('train','val','test')}}
    for name,path in frozen_files.items():
        assert hashlib.sha256(path.read_bytes()).hexdigest()==hashes[name], f'frozen file changed: {name}'
    m=pd.read_csv(PROJECT/'data/interim/manifest_with_split.csv',keep_default_na=False)
    print('Loaded verified frozen manifests; no split files were rewritten.')
else:
    pool=m[m.kept & ~m.pinned].sort_values('clip_id')
    m.loc[m.kept & m.pinned,'split']='train'
    m.loc[pool.index,'split']=splits.two_stage_split(pool,int(CFG['seed']))
    hashes={'manifest_full':splits.write_manifest(m,safe(PROJECT/'data/interim/manifest_full.csv'))}
    for name in ('train','val','test'):
        hashes[name]=splits.write_manifest(m[m.kept & (m.split==name)],safe(P_SPLITS/f'{name}.csv'))
    FROZEN.write_text('Frozen 2026-11-22\nseed: 42\n'+''.join(f'{k}: {v}\n' for k,v in hashes.items()),encoding='utf-8')
    HASH_PATH.write_text(json.dumps(hashes,indent=2)+'\n',encoding='utf-8')
    m.to_csv(safe(PROJECT/'data/interim/manifest_with_split.csv'),index=False,lineterminator='\n')
gates=splits.split_gates(m); display(gates)
assert gates.ok.all(), 'Structural split gate failed; do not freeze or continue.'
display(m[m.kept].groupby(['split','label']).size().unstack(fill_value=0))
print('Decision log draft: Split whole groups using two-stage StratifiedGroupKFold (7 folds, then 6), seed 42; exact-zero representatives are selected by sorted clip_id, preferring silence files; extra_membo_D2 is train-pinned and B_S2_D1 is blocked in consecutive groups of five.')


Loaded verified frozen manifests; no split files were rewritten.


,gate,ok,detail
0,S1 group in one split,True,
1,S2 PCM identity in one split,True,
2,S3 both classes in every split,True,
3,S4 kept pool size,True,10623
4,S5 no exact-zero outside train,True,
5,S6 extra_membo_D2 train only,True,56
6,S7 B_S2_D1 blocks,True,
7,S8 pinned groups train only,True,


label,0,1
split,,
test,1326,180
train,6640,967
val,1325,185


Decision log draft: Split whole groups using two-stage StratifiedGroupKFold (7 folds, then 6), seed 42; exact-zero representatives are selected by sorted clip_id, preferring silence files; extra_membo_D2 is train-pinned and B_S2_D1 is blocked in consecutive groups of five.


In [3]:
# Persist cleaning action counts and the complete split gate table after the freeze.
actions=[]
for reason in ('too_short','exact_zero_duplicate'):
 for label,name in ((0,'unknown'),(1,'drone')): actions.append({'action':'exclude','reason':reason,'flag':'','label':name,'count':int(((m.exclusion_reason==reason)&(m.label==label)).sum())})
for flag in ('near_silent','partial_zero','clipped_flag','dc_flag'):
 for label,name in ((0,'unknown'),(1,'drone')): actions.append({'action':'flag','reason':'','flag':flag,'label':name,'count':int((m[flag].astype(bool)&(m.label==label)).sum())})
pd.DataFrame(actions).to_csv(safe(PROJECT/'data/interim/cleaning_log.csv'),index=False,lineterminator='\n')
saved=json.loads((PROJECT/'results/tables/manifest_hashes.json').read_text(encoding='utf-8'))
hash_ok=all(hashlib.sha256((PROJECT/'data/processed/splits'/f'{name}.csv').read_bytes()).hexdigest()==saved[name] for name in ('train','val','test')) and (PROJECT/'data/processed/splits/FROZEN.txt').is_file()
all_gates=splits.split_gates(m); all_gates=pd.concat([all_gates,pd.DataFrame([{'gate':'S9 manifests hashed and FROZEN.txt written','ok':bool(hash_ok),'detail':'verified against manifest_hashes.json'}])],ignore_index=True)
assert hash_ok, 'frozen split manifest changed'
all_gates.to_csv(safe(PROJECT/'results/tables/split_gates.csv'),index=False,lineterminator='\n')
# Count-only split reports required by the plan. Test is inspected only for counts and identifiers explicitly requested.
k=m[m.kept].copy()
class_counts=(k.groupby(['split','label_name']).size().unstack(fill_value=0).reindex(['train','val','test']))
class_counts['unknown_to_drone_ratio']=(class_counts.get('unknown',0)/class_counts.get('drone',1)).round(3)
class_counts.to_csv(safe(PROJECT/'results/tables/split_class_counts.csv'),lineterminator='\n')
drone=k[k.label==1]
drone_types=pd.crosstab(drone.drone_type,drone.split).reindex(columns=['train','val','test'],fill_value=0)
drone_families=pd.crosstab(drone.family,drone.split).reindex(columns=['train','val','test'],fill_value=0)
family_groups=drone.groupby(['family','split']).group.nunique().unstack(fill_value=0).reindex(columns=['train','val','test'],fill_value=0)
flag_counts=k.groupby('split')[['near_silent','partial_zero','clipped_flag','dc_flag']].sum().reindex(['train','val','test'])
near_silent_drone=k[(k.label==1)&k.near_silent][['clip_id','split']].sort_values('clip_id')
for name,table in [('drone_type_by_split.csv',drone_types),('drone_family_by_split.csv',drone_families),('family_groups_by_split.csv',family_groups),('flag_counts_by_split.csv',flag_counts),('near_silent_drone_split.csv',near_silent_drone)]:
 table.to_csv(safe(PROJECT/'results/tables'/name),lineterminator='\n')
print('Count reports: clips/class/ratio\n',class_counts.to_string())
print('Drone-type clip counts\n',drone_types.to_string())
print('Drone-family clip counts\n',drone_families.to_string())
print('Drone groups by family and split\n',family_groups.to_string())
print('Flag counts by split\n',flag_counts.to_string())
print('Near-silent drone clips and split\n',near_silent_drone.to_string(index=False))
print('cleaning log written; split gates:',all_gates.ok.value_counts().to_dict())
print('Decision log draft: Split whole groups using two-stage StratifiedGroupKFold (7 folds, then 6), seed 42; exact-zero representatives are selected by sorted clip_id, preferring silence files; extra_membo_D2 is train-pinned and B_S2_D1 is blocked in consecutive groups of five.')

Count reports: clips/class/ratio
 label_name  drone  unknown  unknown_to_drone_ratio
split                                             
train         967     6640                   6.867
val           185     1325                   7.162
test          180     1326                   7.367
Drone-type clip counts
 split       train  val  test
drone_type                  
bebop         466  105    95
membo         501   80    85
Drone-family clip counts
 split           train  val  test
family                          
B_S2_D1           231   50    50
Membo_0            90   20    15
Membo_1            70   15     5
Membo_2            45    5    10
extra_membo_D2     56    0     0
mixed_bebop       235   55    45
mixed_membo       240   40    55
Drone groups by family and split
 split           train  val  test
family                          
B_S2_D1             9    2     2
Membo_0            18    4     3
Membo_1            14    3     1
Membo_2             9    1     2
extra_membo_D2  